# 04 — Кластеризация клиентов банка: практический case

Этот ноутбук применяет изученные методы к `bank_churn_dataset.csv`.

## Цель

найти **естественные группы клиентов**, которые похожи по финансовому профилю, использованию банковских услуг, вовлечённости и цифровому поведению.

## 1. Доменная область и гипотезы

Из описания данных можно выделить несколько логических блоков:

**Финансы**
- `balance` — текущий баланс;
- `monthly_ir` — предполагаемый ежемесячный доход;
- `credit_sco` — кредитный рейтинг;
- `risk_score` — расчётный риск.

**Использование продуктов**
- `nums_card`;
- `nums_service`;
- `active_member`.

**Вовлечённость и цифровое поведение**
- `engagement_score`;
- `digital_behavior`;
- `last_active_date`;
- `last_transaction_month`.

**Демография и отношения с банком**
- `age`;
- `gender`;
- `occupation`;
- `tenure_ye`;
- `married`.

Датасет также содержит `customer_segment`, `loyalty_level`, `risk_segment`, `exit` и уже существующий `cluster_group`.

### Гипотезы

**H1.** Более высокий доход может быть связан с большим балансом.

**H2.** Более активные клиенты могут иметь больше используемых сервисов и более высокий engagement score.

**H3.** Давность отношений с банком (`tenure_ye`) может быть связана с лояльностью и числом продуктов.

**H4.** Цифровое поведение может разделять клиентов по характеру взаимодействия с банком.

**H5.** Клиенты с близкими финансовыми и поведенческими характеристиками должны образовывать более устойчивые кластеры, чем группы, построенные только по демографии.

**H6.** Отток (`exit`) потенциально связан с найденными сегментами, но не должен использоваться для построения кластеров, если задача — обнаружить независимую структуру клиентской базы.

> Эти гипотезы — рабочие предположения, а не установленные факты. Их нужно проверить графиками и статистикой.

In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mglearn

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
DATA_PATH = os.path.join("data", "bank_churn_dataset.csv")

df = pd.read_csv(DATA_PATH)
print("Размер:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))

## 2. Первичная проверка качества данных

Перед кластеризацией нельзя считать, что исходный CSV уже подготовлен. Проверим дубликаты, пропуски, диапазоны и типы дат.

In [ ]:
print("Дубликаты строк:", df.duplicated().sum())
print("\nПропуски:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nОписательная статистика числовых признаков:")
display(df.describe(include="number").T)

date_cols = ["last_active_date", "created_date"]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], format="%d/%m/%Y", errors="coerce")

print("\nНекорректно распарсированные даты:")
display(df[date_cols].isna().sum())

## 3. Очистка и конструирование признаков

### Что исключаем

- `id` — идентификатор, а не поведение клиента;
- `full_name` — персональная строка с высокой кардинальностью и без полезной геометрии для расстояния;
- `address`, `origin_province`, `occupation` — можно анализировать отдельно, но прямое one-hot кодирование может заставить расстояние отражать категориальные различия сильнее, чем финансово-поведенческие. Для базовой кластеризации оставим их для интерпретации;
- `exit`, `customer_segment`, `loyalty_level`, `risk_segment` — потенциальные итоговые/контрольные характеристики. Используем их **после** кластеризации для профилирования, а не как входные признаки;
- `cluster_group` — особенно важно исключить: это уже существующая метка кластера и её использование было бы утечкой целевого результата.

### Новые признаки

Из дат построим:
- `account_age_days` — сколько дней прошло от создания аккаунта до последней активности;
- `inactivity_days` — сколько дней прошло от последней активности до общей контрольной даты;
- `balance_log`, `income_log`, `transaction_log` — логарифмируем сильно асимметричные денежные величины.

Логарифм уменьшает влияние очень крупных значений, не превращая их автоматически в выбросы.

In [ ]:
# Контрольная дата: последняя дата, наблюдаемая в данных
reference_date = max(df["last_active_date"].max(), df["created_date"].max())

df["account_age_days"] = (
    df["last_active_date"] - df["created_date"]
).dt.days.clip(lower=0)

df["inactivity_days"] = (
    reference_date - df["last_active_date"]
).dt.days.clip(lower=0)

for src, dst in [
    ("balance", "balance_log"),
    ("monthly_ir", "income_log"),
    ("last_transaction_month", "transaction_log")
]:
    df[dst] = np.log1p(df[src].clip(lower=0))

# bool -> int
df["active_member_num"] = df["active_member"].astype(int)
df["digital_mobile"] = (df["digital_behavior"] == "mobile").astype(int)

feature_cols = [
    "credit_sco", "age", "balance_log", "income_log",
    "tenure_ye", "nums_card", "nums_service", "active_member_num",
    "transaction_log", "engagement_score", "account_age_days",
    "inactivity_days", "risk_score", "digital_mobile"
]

X_raw = df[feature_cols].copy()
print("Количество входных признаков:", len(feature_cols))
display(X_raw.describe().T)

## 4. Проверка выбросов

Выброс не равен автоматически ошибке. В банковских данных крупный баланс или доход может быть реальным клиентом.

Поэтому применим не удаление строк, а **устойчивое преобразование** денежных признаков через `log1p`. Затем стандартизируем признаки.

Это особенно важно для K-means: без стандартизации признак с единицами в десятках/сотнях миллионов может доминировать над признаками вроде `nums_service`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(
    axes,
    ["balance", "monthly_ir", "last_transaction_month"]
):
    sns.histplot(df[c], bins=50, ax=ax)
    ax.set_title(f"Распределение: {c}")
plt.tight_layout()
plt.show()

**Что проверяем:** если распределение имеет длинный правый хвост, среднее может быть сильно выше медианы. Это аргумент в пользу лог-преобразования, а не автоматического удаления верхних процентов наблюдений.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print("Форма матрицы признаков:", X.shape)
print("Среднее первого признака после стандартизации:", X[:, 0].mean())
print("Std первого признака после стандартизации:", X[:, 0].std())

## 5. Feature selection

В базовом варианте оставляем финансовые, продуктовые, поведенческие и временные характеристики.

Не включаем:
- `id`, `full_name`;
- географию и профессию — сначала проверим их как средства интерпретации;
- `exit`, `customer_segment`, `loyalty_level`, `risk_segment`, `cluster_group` — контрольные/итоговые признаки.

Дополнительно можно проверить корреляции входных числовых признаков. Высокая корреляция сама по себе не требует удаления признака: два связанных признака могут нести разные бизнес-смыслы. Решение об исключении принимаем только вместе с проверкой качества кластеров.

In [ ]:
corr = pd.DataFrame(X_raw, columns=feature_cols).corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, center=0)
plt.title("Корреляции признаков, используемых для кластеризации")
plt.tight_layout()
plt.show()

## 6. Разбиение train/test: нужно ли оно?

Для обычной **дескриптивной кластеризации** train/test не является обязательным: мы не предсказываем известную целевую переменную.

Однако hold-out может быть полезен для проверки **устойчивости** найденной структуры. Поэтому здесь:
- строим основную модель на всей доступной выборке;
- дополнительно используем случайные подвыборки для настройки и визуализации;
- для проверки стабильности можно повторить кластеризацию на другой подвыборке.

Это отличается от supervised ML, где train/test обязателен для честной оценки обобщающей способности предсказательной модели.

## 7. 2D-представление исходного пространства

Кластеризация выполняется в многомерном пространстве. Для рисунка сведём стандартизированные признаки к двум главным компонентам PCA.

**Важно:** PCA здесь используется для визуализации, а не заменяет исходное пространство при обучении моделей.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X)

sample_n = min(8000, len(df))
rng = np.random.RandomState(RANDOM_STATE)
sample_idx = rng.choice(len(df), size=sample_n, replace=False)

plt.figure(figsize=(9, 6))
plt.scatter(
    X_2d[sample_idx, 0],
    X_2d[sample_idx, 1],
    s=10,
    alpha=0.35
)
plt.title("Клиенты в 2D-пространстве PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

print("Доля дисперсии PC1+PC2:", pca.explained_variance_ratio_.sum())

## 8. Проверка доменных гипотез

### H1: доход и баланс

Проверим связь на выборке, чтобы график оставался читаемым.

In [ ]:
plot_df = df.sample(sample_n, random_state=RANDOM_STATE)

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=plot_df,
    x="monthly_ir",
    y="balance",
    alpha=0.35,
    s=18
)
plt.title("Доход и баланс клиента")
plt.xlabel("Предполагаемый месячный доход")
plt.ylabel("Баланс")
plt.show()

Если облако имеет выраженный восходящий тренд, H1 получает визуальную поддержку. Если связь слабая, это тоже важный результат: баланс нельзя объяснять только доходом.

### H2: активность, сервисы и вовлечённость

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=plot_df,
    x="engagement_score",
    y="nums_service",
    hue="active_member",
    alpha=0.4,
    s=18
)
plt.title("Вовлечённость и количество услуг")
plt.xlabel("Engagement score")
plt.ylabel("Количество услуг")
plt.show()

### H3/H4: стаж и цифровое поведение

Проверим распределения стажа по цифровому поведению и связь с продуктовым использованием.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(data=plot_df, x="digital_behavior", y="tenure_ye", ax=axes[0])
axes[0].set_title("Стаж по цифровому поведению")

sns.boxplot(data=plot_df, x="digital_behavior", y="nums_service", ax=axes[1])
axes[1].set_title("Количество услуг по цифровому поведению")

plt.tight_layout()
plt.show()

## 9. K-means: подбор количества кластеров

Не будем заранее утверждать, что клиентов нужно разделить, например, на 4 группы.

Для `K=2..8` вычислим inertia и silhouette на случайной подвыборке. Это снижает вычислительную стоимость silhouette на 80 000 объектов.

In [ ]:
eval_n = min(10000, len(df))
eval_idx = rng.choice(len(df), size=eval_n, replace=False)
X_eval = X[eval_idx]

k_results = []
for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=10,
        random_state=RANDOM_STATE
    )
    lab = model.fit_predict(X_eval)
    k_results.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_eval, lab)
    })

k_results = pd.DataFrame(k_results)
display(k_results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(k_results["k"], k_results["inertia"], marker="o")
axes[0].set_title("Elbow: inertia")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")

axes[1].plot(k_results["k"], k_results["silhouette"], marker="o")
axes[1].set_title("Silhouette")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Score")

plt.tight_layout()
plt.show()

best_k_silhouette = int(
    k_results.loc[k_results["silhouette"].idxmax(), "k"]
)
print("K с максимальным silhouette:", best_k_silhouette)

**Правило выбора:** `best_k_silhouette` — не автоматически окончательный `K`. Сопоставляем его с elbow и с тем, насколько осмысленны профили полученных групп.

## 10. Финальный K-means и K-means++

`K-means++` задаётся `init="k-means++"`. Это не другой принцип оптимизации кластеров, а более продуманная стратегия выбора начальных центроидов.

Сначала обучим K-means++ на всех клиентах.

In [ ]:
kmeans_pp = KMeans(
    n_clusters=best_k_silhouette,
    init="k-means++",
    n_init=20,
    random_state=RANDOM_STATE
)
labels_kmeans = kmeans_pp.fit_predict(X)

df["cluster_kmeans_pp"] = labels_kmeans

print("Размеры кластеров:")
display(df["cluster_kmeans_pp"].value_counts().sort_index())

## 11. 2D визуализация кластеров

Это визуализация **многомерной кластеризации через PCA**, поэтому возможна ситуация, когда на 2D графике кластеры выглядят пересекающимися, хотя в полном пространстве они разделены.

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(
    X_2d[sample_idx, 0],
    X_2d[sample_idx, 1],
    c=labels_kmeans[sample_idx],
    s=12,
    alpha=0.5
)
plt.title("K-means++: кластеры в PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## 12. MiniBatchKMeans как масштабируемая альтернатива

Сравним качество MiniBatchKMeans с обычным K-means. На больших данных небольшая потеря качества может быть оправдана существенным ускорением.

In [ ]:
mbk = MiniBatchKMeans(
    n_clusters=best_k_silhouette,
    batch_size=1024,
    n_init=10,
    random_state=RANDOM_STATE
)
labels_mbk = mbk.fit_predict(X)

comparison = pd.DataFrame({
    "algorithm": ["KMeans++", "MiniBatchKMeans"],
    "inertia": [kmeans_pp.inertia_, mbk.inertia_],
    "silhouette_sample": [
        silhouette_score(X[eval_idx], labels_kmeans[eval_idx]),
        silhouette_score(X[eval_idx], labels_mbk[eval_idx])
    ]
})
display(comparison)

## 13. Иерархическая кластеризация

Для 80 000 объектов полная агломеративная кластеризация может быть дорогой по времени и памяти. Поэтому используем репрезентативную подвыборку.

Это подходит для:
- изучения структуры;
- сравнения с K-means;
- построения дендрограммы.

Для production-задачи размер подвыборки и алгоритм следует выбирать с учётом доступной памяти.

In [ ]:
hier_n = min(4000, len(df))
hier_idx = rng.choice(len(df), size=hier_n, replace=False)

hier = AgglomerativeClustering(
    n_clusters=best_k_silhouette,
    linkage="ward"
)
hier_labels = hier.fit_predict(X[hier_idx])

print("Silhouette hierarchical:",
      silhouette_score(X[hier_idx], hier_labels))

plt.figure(figsize=(9, 6))
plt.scatter(
    X_2d[hier_idx, 0],
    X_2d[hier_idx, 1],
    c=hier_labels,
    s=15,
    alpha=0.55
)
plt.title("Иерархическая кластеризация: PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## 14. DBSCAN

DBSCAN не требует задавать число кластеров, но требует подобрать `eps` и `min_samples`.

В многомерном пространстве `eps` особенно чувствителен к масштабу, поэтому выше мы стандартизировали признаки.

Для демонстрации используем подвыборку.

In [ ]:
db_n = min(10000, len(df))
db_idx = rng.choice(len(df), size=db_n, replace=False)

# Значение eps здесь стартовое, а не универсально правильное.
dbscan = DBSCAN(eps=0.9, min_samples=20)
db_labels = dbscan.fit_predict(X[db_idx])

n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
noise_share = np.mean(db_labels == -1)

print("Кластеров DBSCAN:", n_db_clusters)
print("Доля шума:", noise_share)

plt.figure(figsize=(9, 6))
plt.scatter(
    X_2d[db_idx, 0],
    X_2d[db_idx, 1],
    c=db_labels,
    s=12,
    alpha=0.55
)
plt.title("DBSCAN: PCA 2D, -1 = шум")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

### Как подбирать `eps`

Вместо угадывания полезно исследовать расстояния до `k`-го соседа и искать характерный «перегиб» кривой. Параметр зависит от масштаба и структуры именно вашего датасета.

Также нельзя выбирать `eps` только потому, что картинка стала красивой: нужно смотреть на долю шума, число кластеров и устойчивость результата.

In [ ]:
from sklearn.neighbors import NearestNeighbors

k_neighbors = 20
nn = NearestNeighbors(n_neighbors=k_neighbors).fit(X[db_idx])
distances, _ = nn.kneighbors(X[db_idx])
kdist = np.sort(distances[:, -1])

plt.figure(figsize=(10, 5))
plt.plot(kdist)
plt.title(f"{k_neighbors}-NN distance curve для подбора eps")
plt.xlabel("Отсортированный объект")
plt.ylabel(f"Расстояние до {k_neighbors}-го соседа")
plt.show()

## 15. HDBSCAN

HDBSCAN особенно интересен, когда плотность разных групп различается.

Код ниже совместим с двумя распространёнными вариантами установки: `sklearn.cluster.HDBSCAN` в новых версиях scikit-learn или внешний пакет `hdbscan`.

In [ ]:
try:
    from sklearn.cluster import HDBSCAN
    hdb = HDBSCAN(min_cluster_size=50)
    hdb_labels = hdb.fit_predict(X[db_idx])
    hdb_source = "sklearn"
except ImportError:
    # При необходимости:
    # pip install hdbscan
    import hdbscan
    hdb = hdbscan.HDBSCAN(min_cluster_size=50)
    hdb_labels = hdb.fit_predict(X[db_idx])
    hdb_source = "hdbscan package"

n_hdb_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
hdb_noise = np.mean(hdb_labels == -1)

print("Реализация:", hdb_source)
print("Кластеров HDBSCAN:", n_hdb_clusters)
print("Доля шума:", hdb_noise)

plt.figure(figsize=(9, 6))
plt.scatter(
    X_2d[db_idx, 0],
    X_2d[db_idx, 1],
    c=hdb_labels,
    s=12,
    alpha=0.55
)
plt.title("HDBSCAN: PCA 2D, -1 = шум")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## 16. Сводное сравнение алгоритмов

Сравниваем не только одно число, но:
- количество кластеров;
- долю шума;
- silhouette, если он корректно применим;
- визуальную устойчивость;
- бизнес-интерпретируемость.

Для DBSCAN/HDBSCAN шум (`-1`) обычно исключают из silhouette, если задача — оценить только найденные плотные группы.

In [ ]:
def safe_silhouette(X_part, labels):
    mask = labels != -1
    valid_labels = labels[mask]
    if len(np.unique(valid_labels)) < 2:
        return np.nan
    return silhouette_score(X_part[mask], valid_labels)

summary = pd.DataFrame([
    {
        "algorithm": "KMeans++",
        "n_clusters": len(np.unique(labels_kmeans)),
        "noise_share": 0,
        "silhouette": silhouette_score(X[eval_idx], labels_kmeans[eval_idx])
    },
    {
        "algorithm": "MiniBatchKMeans",
        "n_clusters": len(np.unique(labels_mbk)),
        "noise_share": 0,
        "silhouette": silhouette_score(X[eval_idx], labels_mbk[eval_idx])
    },
    {
        "algorithm": "Hierarchical (sample)",
        "n_clusters": len(np.unique(hier_labels)),
        "noise_share": 0,
        "silhouette": silhouette_score(X[hier_idx], hier_labels)
    },
    {
        "algorithm": "DBSCAN (sample)",
        "n_clusters": n_db_clusters,
        "noise_share": noise_share,
        "silhouette": safe_silhouette(X[db_idx], db_labels)
    },
    {
        "algorithm": "HDBSCAN (sample)",
        "n_clusters": n_hdb_clusters,
        "noise_share": hdb_noise,
        "silhouette": safe_silhouette(X[db_idx], hdb_labels)
    }
])
display(summary)

## 17. Интерпретация кластеров K-means++

После получения меток недостаточно сказать «кластер 0 хороший». Нужно понять, **какие клиенты** в него попали.

Используем:
1. размер каждого кластера;
2. медианы числовых признаков;
3. категориальные распределения;
4. контрольные поля `exit`, `customer_segment`, `loyalty_level`, `risk_segment`.

Контрольные поля не участвовали в обучении, поэтому их можно использовать для постфактум-профилирования.

In [ ]:
profile_features = [
    "credit_sco", "age", "balance", "monthly_ir",
    "tenure_ye", "nums_card", "nums_service",
    "engagement_score", "risk_score",
    "account_age_days", "inactivity_days",
    "last_transaction_month"
]

cluster_profile = (
    df.groupby("cluster_kmeans_pp")[profile_features]
      .median()
      .round(2)
)

cluster_sizes = (
    df["cluster_kmeans_pp"]
      .value_counts()
      .sort_index()
      .rename("count")
      .to_frame()
)
cluster_sizes["share"] = cluster_sizes["count"] / len(df)

display(cluster_sizes)
display(cluster_profile)

### Профили категориальных признаков

Вместо абсолютных значений полезно смотреть доли категорий внутри каждого кластера.

In [ ]:
for c in [
    "gender", "occupation", "digital_behavior",
    "customer_segment", "loyalty_level", "risk_segment", "exit"
]:
    print(f"\n--- {c} ---")
    table = pd.crosstab(
        df["cluster_kmeans_pp"],
        df[c],
        normalize="index"
    ).round(3)
    display(table)

## 18. Проверка связи кластеров с оттоком

`exit` не использовался как входной признак. Поэтому сейчас его можно рассматривать как **внешнюю проверку**: отличаются ли найденные сегменты по фактическому индикатору ухода.

Это не превращает кластеризацию в supervised learning и не доказывает причинность.

Если кластеры имеют разные значения `exit`, это означает лишь, что сегментация связана с оттоком.

In [ ]:
exit_by_cluster = pd.crosstab(
    df["cluster_kmeans_pp"],
    df["exit"],
    normalize="index"
).round(3)

display(exit_by_cluster)

## 19. Использование `cluster_group`

В датасете уже существует поле `cluster_group`, описанное как метка кластера клиентов.

Мы **не использовали его для обучения**, иначе получилась бы утечка.

Но его можно сравнить с нашими результатами как отдельную внешнюю контрольную разметку. Для этого подходят ARI и NMI

Важно: совпадение с `cluster_group` не означает автоматически, что найденная сегментация «истинная». Это лишь показывает согласованность с уже существующей меткой.

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(df["cluster_group"], df["cluster_kmeans_pp"])
nmi = normalized_mutual_info_score(df["cluster_group"], df["cluster_kmeans_pp"])

print("ARI относительно cluster_group:", ari)
print("NMI относительно cluster_group:", nmi)

## 20. Итоговая интерпретация

После выполнения ноутбука итоговые кластеры нужно описывать **не их номером, а профилем**.

Хороший шаблон:

> **Кластер X — «активные высокововлечённые клиенты».**  
> У них выше/ниже медианный `engagement_score`, больше/меньше `nums_service`, отличается `digital_behavior`, а также наблюдается такой-то профиль баланса и дохода. Доля клиентов с `exit=True` составляет ... . Поэтому потенциальная бизнес-интерпретация — ...

Нельзя делать вывод:

> «Кластер 2 — плохие клиенты»

только на основании одного `risk_score` или `exit`. Кластер — многомерная группа, и интерпретация должна опираться на совокупность признаков.

### Финальный чек-лист

- [x] Проверены типы, пропуски и дубликаты.
- [x] Денежные признаки приведены к более удобному масштабу.
- [x] Убраны идентификаторы и потенциальные leakage-признаки.
- [x] Созданы временные признаки.
- [x] Масштабирование выполнено перед алгоритмами, основанными на расстояниях.
- [x] Проверены доменные гипотезы.
- [x] `K` выбран на основании нескольких критериев.
- [x] Сравнены K-means++, MiniBatchKMeans, hierarchical, DBSCAN и HDBSCAN.
- [x] Построены 2D-визуализации через PCA.
- [x] Выполнено профилирование кластеров.
- [x] Внешние labels используются только для независимой проверки.